To utilize the Azure Machine Learning service for building and executing machine learning workflows, you'll need to install the Azure Machine Learning SDK for Python. This SDK allows seamless integration with Azure services and provides tools for training and deploying machine learning models in the cloud. 

In [ ]:
! pip install azureml-sdk

In [ ]:
import os
import json
import requests

from azureml.core import Workspace
from azureml.core.model import Model
from azureml.core.environment import Environment
from azureml.core.conda_dependencies import CondaDependencies
from azureml.core.model import InferenceConfig
from azureml.core.webservice import AciWebservice, Webservice
from azure.mgmt.resource import ResourceManagementClient
from azure.identity import DefaultAzureCredential

**Instruction**: [Refer to sample_azure_configs](sample_azure_configs.json) file and create an "azure_configs.json" file with your own details accordingly.

In [ ]:
config_file_path = "azure_configs.json"

with open(config_file_path, 'r') as file:
    data = json.load(file)

subscription_id = data["subscription_id"]
resource_group = data["resource_group"]
workspace_name = data["workspace_name"]
region = data["region"]

**Note**: Before executing this cell, ensure that a resource group with the same name as mentioned in your "azure_configs.json" file has been created. You can create it using the Azure portal. In this cell, an ML Workspace is created. If a Workspace already exists, it will be used.

In [ ]:
try:
    ws = Workspace(subscription_id=subscription_id, resource_group=resource_group, workspace_name=workspace_name)
    print(f'Workspace {workspace_name} found.')
except Exception as e:
    ws = Workspace.create(name=workspace_name,
                          subscription_id=subscription_id,
                          resource_group=resource_group,
                          location=region)
    print(f'Workspace {workspace_name} created.')


In this cell two variables are defined:

1. `model_path`: This variable contains the path to the pre-trained PyTorch model file named "RRDB_ESRGAN_x4.pth". This model file is assumed to be located in the "/models" directory.
2. `model_name`: This variable specifies the name to be assigned to the registered model in the Azure Machine Learning workspace. In this case, the model will be registered with the name "image-super-resolution".

These variables are typically used in Azure Machine Learning workflows to specify the location of the model file and the name under which the model will be registered in the workspace.

In [ ]:
model_path = '/models/RRDB_ESRGAN_x4.pth'
model_name='image-super-resolution'

In this cell following tasks are performed:

1. Retrieves a list of registered models from the specified Azure Machine Learning workspace (`ws`).
2. Checks whether a model with the name specified in the variable `model_name` already exists in the workspace.
3. If a model with the same name is found, it prints a message indicating that the model already exists in the workspace.
4. If no model with the same name is found, it registers the model using the `Model.register` method. It specifies the path to the model file (`model_path`), the desired name for the registered model (`model_name`), and the Azure Machine Learning workspace (`ws`). After successful registration, it prints a message confirming the registration of the model.

In [ ]:
registered_models = Model.list(workspace=ws)
model_already_registered = any(model.name == model_name for model in registered_models)
registered_model = None

if model_already_registered:
    print(f"Model '{model_name}' already exists in the workspace.")
    registered_model = next(model for model in registered_models if model.name == model_name)
else:
    registered_model = Model.register(model_path=model_path, model_name=model_name, workspace=ws)
    print(f"Model '{model_name}' registered successfully.")


This cell creates a Conda environment named 'my-conda-env' for a Python project. The environment is configured with specific packages and their dependencies using CondaDependencies.

- `conda_packages`: This variable contains a list of Conda packages to be installed in the environment. The packages include Python 3.8, PyTorch, TorchVision, NumPy, and Pillow.

- `conda_deps`: Using the `CondaDependencies.create` method, a CondaDependencies object is created with the specified Conda packages.

- The Conda environment is then configured with the created CondaDependencies object, ensuring that the required packages are installed in the environment.

This setup ensures that the necessary Python and library dependencies are available within the Conda environment for the project.


In [ ]:
conda_env = Environment('my-conda-env')

conda_packages = [
    'python=3.8',
    'pytorch',
    'torchvision',
    'numpy',
    'pillow'
]

conda_deps = CondaDependencies.create(conda_packages=conda_packages)

conda_env.python.conda_dependencies = conda_deps

This cell defines an InferenceConfig object, which is used to configure the environment and entry script for model inference in Azure Machine Learning.

- `source_directory`: Specifies the directory containing the inference code. In this case, it is set to 'src', indicating that the inference script and any associated files are located in the 'src' directory.

- `entry_script`: Specifies the entry point script for model inference. Here, 'score.py' is specified as the entry script, indicating that this script contains the code for model scoring or prediction.

- `environment`: Specifies the Conda environment to be used for inference. The previously defined Conda environment named 'my-conda-env' (stored in the variable `conda_env`) is assigned to the environment parameter. This ensures that the model is run in an environment with the required dependencies specified in the Conda environment.

Together, this InferenceConfig object provides the necessary configuration for deploying and running model inference in Azure Machine Learning.

In [ ]:
inference_config = InferenceConfig(source_directory='src', entry_script='score.py', environment=conda_env)

This cell creates an ACI (Azure Container Instances) deployment configuration object using the `AciWebservice.deploy_configuration` method. ACI is a serverless compute service in Azure that enables you to deploy containers without managing the underlying infrastructure.

- `cpu_cores`: Specifies the number of CPU cores to allocate for the deployed ACI container. Here, it is set to 1, indicating that the deployed container will have access to one CPU core.

- `memory_gb`: Specifies the amount of memory (RAM) to allocate for the deployed ACI container, measured in gigabytes (GB). In this case, it is set to 1 GB, indicating that the deployed container will have access to 1 GB of memory.

These configuration settings define the resource allocation for the deployed ACI container, ensuring that it has sufficient computational resources to execute the deployed model inference service.

In [ ]:
aci_config = AciWebservice.deploy_configuration(cpu_cores=3, memory_gb=6)

This cell performs the following tasks:

1. Checks if a service named 'image-super-resolution' already exists in the Azure Machine Learning workspace.
2. If the service already exists, it deletes the existing service.
3. Deploys a new service named 'image-super-resolution' using the registered model, inference configuration, and ACI deployment configuration.
4. Waits for the deployment to complete and prints the deployment output.
5. Prints a message indicating that the service 'image-super-resolution' has been deployed successfully.

This code is responsible for managing the deployment of the machine learning model as a web service in the Azure Machine Learning workspace. It ensures that any existing service with the same name is deleted before deploying the new service.

In [ ]:
existing_services = Webservice.list(workspace=ws)
service_already_exists = any(service.name == 'image-super-resolution' for service in existing_services)

if service_already_exists:
    existing_service = Webservice(workspace=ws, name='image-super-resolution')
    existing_service.delete()
    print("Existing service 'image-super-resolution' deleted.")

service = Model.deploy(workspace=ws,
                       name='image-super-resolution',
                       models=[registered_model],
                       inference_config=inference_config,
                       deployment_config=aci_config)
service.wait_for_deployment(show_output=True)
print("Service 'image-super-resolution' deployed successfully.")

The `scoring_uri` variable contains the URI endpoint for accessing the deployed machine learning service. It is obtained from the `service` object, which represents the deployed service in the Azure Machine Learning workspace.

The scoring URI is the endpoint that clients can use to interact with the deployed model for making predictions. It typically accepts HTTP requests containing input data, processes the data using the deployed model, and returns the predictions or results.

The `scoring_uri` is crucial for the backend server of the application as it specifies the endpoint where the server should send requests to obtain predictions from the deployed model.

In [ ]:
scoring_uri = service.scoring_uri
# print(scoring_uri)

Following code is for testing the endpoint, the following code will work only if the lines of code using "ast" library are uncommented in [score.py](./src/score.py).

In [ ]:
import io
import json
import base64
import requests
from PIL import Image
import os
import ast

This cell performs the following tasks:

1. Reads the contents of an image file.
2. Encodes the image data using Base64 encoding.
3. Prepares the image data to be sent in a POST request by packaging it into a JSON object.
4. Sends a POST request to the scoring URI endpoint, which is assumed to be stored in the variable `scoring_uri`.
5. Checks if the request was successful by verifying the HTTP status code of the response.
6. If the request was successful (status code 200), decodes the received JSON data to extract the encoded enhanced image data.
7. Decodes the encoded enhanced image data using Base64 decoding.
8. Saves the decoded enhanced image data to a file named "enhanced_image.png".
9. Prints a success message indicating that the enhanced image has been saved successfully, along with the path to the saved image file.
10. If the request was not successful, prints an error message containing the response text.

In [ ]:
# Open the image file and read its contents
with open("your_image_file_name", "rb") as image_file:
    raw_image_data = image_file.read()

# Encode the image data using Base64
encoded_image_data = base64.b64encode(raw_image_data).decode('utf-8')

# Prepare the data to be sent in the POST request
data_to_send = {"image": encoded_image_data}
json_data = json.dumps(data_to_send)

# Make the POST request to the scoring URI
response = requests.post(scoring_uri, json=json_data)

# Check if the request was successful
if response.status_code == 200:
    # Decode the received JSON data
    unescaped_data = ast.literal_eval(response.text)
    reveived_data = json.loads(unescaped_data)
    encoded_enhanced_img_data = reveived_data["image"]

    # Decode the received enhanced image data
    enhanced_img_data = base64.b64decode(encoded_enhanced_img_data)

    # Save the enhanced image
    output_image_path = "enhanced_image.png"
    with open(output_image_path, "wb") as output_image_file:
        output_image_file.write(enhanced_img_data)

    print("Enhanced image saved successfully:", output_image_path)
else:
    print("Error:", response.text)
